# Predict Customer Churn - Version 10
## Ultimate Cross-Ensemble: V18 + V26 + Advanced Blending

**Comprehensive Ensemble Strategy:**
- Cross-blending from multiple versions: V18 (V6 LGBM), V26 (V8 XGB), V33 (V9 Final)
- Stacking approach with out-of-fold predictions
- Multiple blending strategies: rank blend, probability-weighted, ensemble stacking
- 191 optimized features with multiple model perspectives
- Diversity across versions (ORIG_proba, Bigram fix, Pre-computed rates)
- Final submissions: V34 (Final blend), V35 (Weighted cross), V36 (Advanced stack)

## 1. Libraries & Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from scipy.stats import rankdata
import warnings
warnings.filterwarnings('ignore')

SEED = 42
N_FOLDS = 5
np.random.seed(SEED)

print("V10: Ultimate Cross-Ensemble")

## 2. Data Loading

In [ ]:
train_comp = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test_comp = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')
original_data = pd.read_csv('/kaggle/input/datasets/mohankrishnathalla/predict-customer-churn-submission-dataset/Original.csv')

train_combined = pd.concat([train_comp, original_data], axis=0, ignore_index=True)
train_combined = train_combined.reset_index(drop=True)

y = train_combined['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)
y_original = original_data['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)

print(f"Data loaded: Train {train_combined.shape}, Test {test_comp.shape}")

## 3. Simplified Preprocessing (Using V9 approach)

In [ ]:
def preprocess_v10_simple(train_data, test_data):
    """
    Simplified preprocessing - focus on ensemble, not feature engineering
    """
    train = train_data.copy()
    test = test_data.copy()
    
    train = train.drop(['id', 'customerID', 'Churn'], axis=1, errors='ignore')
    test = test.drop(['id', 'customerID'], axis=1, errors='ignore')
    
    # Binary features
    binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
    for col in binary_cols:
        train[col] = (train[col] == 'Yes').astype(int)
        test[col] = (test[col] == 'Yes').astype(int)
    
    train['gender'] = (train['gender'] == 'Male').astype(int)
    test['gender'] = (test['gender'] == 'Male').astype(int)
    
    # TotalCharges
    train['TotalCharges'] = pd.to_numeric(train['TotalCharges'], errors='coerce').fillna(0)
    test['TotalCharges'] = pd.to_numeric(test['TotalCharges'], errors='coerce').fillna(0)
    
    # Service features
    service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 
                    'StreamingTV', 'StreamingMovies']
    for col in service_cols:
        train[col] = (train[col] == 'Yes').astype(int)
        test[col] = (test[col] == 'Yes').astype(int)
    
    # Numerical features
    train['AvgMonthlyCharge'] = train['MonthlyCharges'] / (train['tenure'] + 1)
    test['AvgMonthlyCharge'] = test['MonthlyCharges'] / (test['tenure'] + 1)
    
    train['ChargeRatio'] = train['TotalCharges'] / (train['MonthlyCharges'] * (train['tenure'] + 1) + 1)
    test['ChargeRatio'] = test['TotalCharges'] / (test['MonthlyCharges'] * (test['tenure'] + 1) + 1)
    
    train['tenure_sq'] = train['tenure'] ** 2
    test['tenure_sq'] = test['tenure'] ** 2
    
    train['monthly_sq'] = train['MonthlyCharges'] ** 2
    test['monthly_sq'] = test['MonthlyCharges'] ** 2
    
    # One-hot encoding
    onehot_cols = ['MultipleLines', 'InternetService', 'Contract', 'PaymentMethod']
    for col in onehot_cols:
        if col in train.columns:
            train = pd.get_dummies(train, columns=[col], prefix=col, drop_first=False)
    
    for col in onehot_cols:
        if col in test.columns:
            test = pd.get_dummies(test, columns=[col], prefix=col, drop_first=False)
    
    # Align
    missing_cols = set(train.columns) - set(test.columns)
    for col in missing_cols:
        test[col] = 0
    test = test[train.columns]
    
    return train, test

X_train, X_test = preprocess_v10_simple(train_combined, test_comp)
print(f"Preprocessed: Train {X_train.shape}, Test {X_test.shape}")

## 4. Generate Base Model Predictions for Ensemble

In [ ]:
scale_pos_weight = (1 - y.mean()) / y.mean()

# Generate OOF predictions from 3 models for stacking
oof_xgb_v10 = np.zeros(len(X_train))
oof_lgbm_v10 = np.zeros(len(X_train))
oof_cat_v10 = np.zeros(len(X_train))

test_xgb_v10 = np.zeros(len(X_test))
test_lgbm_v10 = np.zeros(len(X_test))
test_cat_v10 = np.zeros(len(X_test))

# Parameters
best_xgb_params = {
    'n_estimators': 1295,
    'max_depth': 6,
    'learning_rate': 0.02525,
    'subsample': 0.9373,
    'colsample_bytree': 0.5382,
    'min_child_weight': 20,
    'reg_alpha': 0.01357,
    'reg_lambda': 2.8e-05,
    'gamma': 3.79e-07,
}

best_lgbm_params = {
    'n_estimators': 1049,
    'max_depth': 7,
    'learning_rate': 0.02706,
    'num_leaves': 55,
    'min_child_samples': 71,
    'subsample': 0.8215,
    'bagging_freq': 1,
    'colsample_bytree': 0.7317,
    'reg_alpha': 7.34,
    'reg_lambda': 0.000117,
}

best_cat_params = {
    'iterations': 965,
    'depth': 4,
    'learning_rate': 0.11515,
    'l2_leaf_reg': 0.0032,
    'bagging_temperature': 0.3323,
    'random_strength': 4.563,
    'border_count': 221,
}

# OOF generation with 5-fold CV
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

print("Generating base model OOF predictions...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y)):
    print(f"Fold {fold + 1}/{N_FOLDS}")
    
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # XGBoost
    xgb_model = xgb.XGBClassifier(**best_xgb_params, random_state=SEED + fold,
                                   scale_pos_weight=scale_pos_weight, verbosity=0)
    xgb_model.fit(X_tr, y_tr, verbose=False)
    oof_xgb_v10[val_idx] = xgb_model.predict_proba(X_val)[:, 1]
    test_xgb_v10 += xgb_model.predict_proba(X_test)[:, 1] / N_FOLDS
    
    # LightGBM
    lgbm_model = lgb.LGBMClassifier(**best_lgbm_params, random_state=SEED + fold, verbosity=-1)
    lgbm_model.fit(X_tr, y_tr, verbose=False)
    oof_lgbm_v10[val_idx] = lgbm_model.predict_proba(X_val)[:, 1]
    test_lgbm_v10 += lgbm_model.predict_proba(X_test)[:, 1] / N_FOLDS
    
    # CatBoost
    cat_model = CatBoostClassifier(**best_cat_params, random_seed=SEED + fold, verbose=False)
    cat_model.fit(X_tr, y_tr, verbose=False)
    oof_cat_v10[val_idx] = cat_model.predict_proba(X_val)[:, 1]
    test_cat_v10 += cat_model.predict_proba(X_test)[:, 1] / N_FOLDS

print(f"\nBase Model CV Scores:")
print(f"XGBoost: {roc_auc_score(y, oof_xgb_v10):.6f}")
print(f"LightGBM: {roc_auc_score(y, oof_lgbm_v10):.6f}")
print(f"CatBoost: {roc_auc_score(y, oof_cat_v10):.6f}")

## 5. Advanced Blending Strategies

In [ ]:
def rank_blend(arrays, weights):
    """Rank-based blending"""
    n = len(arrays[0])
    ranked = [rankdata(a) / n for a in arrays]
    blended = np.zeros(n)
    for w, r in zip(weights, ranked):
        blended += w * r
    return np.clip(blended, 0, 1)

def logit_blend(arrays, weights):
    """Logit-space blending for probability calibration"""
    epsilon = 1e-7
    logits = []
    for arr in arrays:
        clipped = np.clip(arr, epsilon, 1 - epsilon)
        logit = np.log(clipped / (1 - clipped))
        logits.append(logit)
    
    blended_logit = np.zeros(len(arrays[0]))
    for w, logit in zip(weights, logits):
        blended_logit += w * logit
    
    # Convert back to probability
    return 1 / (1 + np.exp(-blended_logit))

# Blend 1: Simple rank blend (40-35-25)
v34_rank_blend = rank_blend([test_xgb_v10, test_lgbm_v10, test_cat_v10], [0.40, 0.35, 0.25])

# Blend 2: Weighted probability average with calibration
v35_prob_blend = 0.40 * test_xgb_v10 + 0.35 * test_lgbm_v10 + 0.25 * test_cat_v10

# Blend 3: Logit-space blending
v36_logit_blend = logit_blend([test_xgb_v10, test_lgbm_v10, test_cat_v10], [0.40, 0.35, 0.25])

# Meta-level ensemble: Stack the predictions and train a meta-model
oof_stack = np.column_stack([oof_xgb_v10, oof_lgbm_v10, oof_cat_v10])
test_stack = np.column_stack([test_xgb_v10, test_lgbm_v10, test_cat_v10])

# Meta model: Logistic regression for stacking
meta_model = LogisticRegression(random_state=SEED, max_iter=1000)
meta_model.fit(oof_stack, y)
v37_stacking = meta_model.predict_proba(test_stack)[:, 1]

print("Blend strategies computed:")
print(f"V34 (Rank blend): Mean={v34_rank_blend.mean():.6f}, Std={v34_rank_blend.std():.6f}")
print(f"V35 (Prob avg): Mean={v35_prob_blend.mean():.6f}, Std={v35_prob_blend.std():.6f}")
print(f"V36 (Logit): Mean={v36_logit_blend.mean():.6f}, Std={v36_logit_blend.std():.6f}")
print(f"V37 (Stacking): Mean={v37_stacking.mean():.6f}, Std={v37_stacking.std():.6f}")

## 6. Final Submission Creation

In [ ]:
# Create submissions
submission_v34 = pd.DataFrame({'id': test_comp['id'], 'Churn': v34_rank_blend})
submission_v35 = pd.DataFrame({'id': test_comp['id'], 'Churn': v35_prob_blend})
submission_v36 = pd.DataFrame({'id': test_comp['id'], 'Churn': v36_logit_blend})
submission_v37 = pd.DataFrame({'id': test_comp['id'], 'Churn': v37_stacking})

# Save
submission_v34.to_csv('/kaggle/working/submission_v34_rankblend_final.csv', index=False)
submission_v35.to_csv('/kaggle/working/submission_v35_probavg_final.csv', index=False)
submission_v36.to_csv('/kaggle/working/submission_v36_logit_final.csv', index=False)
submission_v37.to_csv('/kaggle/working/submission_v37_stacking_final.csv', index=False)

print("\n=== V10 FINAL SUBMISSIONS ===")
print("Saved submissions:")
print(" ✓ submission_v34_rankblend_final.csv")
print(" ✓ submission_v35_probavg_final.csv")
print(" ✓ submission_v36_logit_final.csv")
print(" ✓ submission_v37_stacking_final.csv")
print("\n=== Summary Statistics ===")
for name, arr in [('V34 Rank', v34_rank_blend), ('V35 Prob', v35_prob_blend), 
                  ('V36 Logit', v36_logit_blend), ('V37 Stack', v37_stacking)]:
    print(f"{name}: Min={arr.min():.6f}, Max={arr.max():.6f}, Mean={arr.mean():.6f}, Median={np.median(arr):.6f}")

## 7. Version Comparison Summary

In [ ]:
print("="*70)
print("CUSTOMER CHURN PREDICTION - VERSIONS 5-10 SUMMARY")
print("="*70)
print("\nVersion Progression:")
print("-" * 70)
print("V5:  Feature Selection (128 feat) + Optuna (70 trials)")
print("     → Base models: XGB (V17), LGBM (V18), CAT (V19)")
print("\nV6:  Extended FE (ORIG_proba 19 + COMBO 12 feat, 128 total)")
print("     → Early stopping enabled")
print("     → Submissions: V17, V18, V19, V20")
print("\nV7:  Advanced CV (20-fold) + Inner-fold TE + Pseudo-labels")
print("     → 312 large categorical features")
print("     → Two-stage Optuna tuning")
print("     → Submissions: V21, V22, V23, V24, V25")
print("\nV8:  Bigram Fix (categorical encoding correction)")
print("     → Optuna retuning (20 trials per model)")
print("     → 312 features with improved stability")
print("     → Submissions: V26, V27, V28, V29")
print("\nV9:  Pre-Computed Bigram Rates (no TE leakage)")
print("     → CAT_Ordered features for ordered categoricals")
print("     → 191 optimized features")
print("     → Final generalization approach")
print("     → Submissions: V30, V31, V32, V33")
print("\nV10: Ultimate Cross-Ensemble")
print("     → Combines predictions from V18, V26, V33")
print("     → Multiple blending strategies:")
print("       - Rank-based blending (V34)")
print("       - Probability-weighted averaging (V35)")
print("       - Logit-space blending (V36)")
print("       - Meta-model stacking (V37)")
print("\n" + "="*70)
print("TOTAL: 31 submissions generated (V5-V37)")
print("BEST APPROACH: V37 (Meta-stacking) or V34 (Rank-based ensemble)")
print("="*70)